# 13 Model Training — Baseline Models

This notebook is for baseline model training only.

- Goal: train simple baseline models for fraud detection.
- Main focus: recall, because missing a fraudulent transaction is costly.
- Precision is also important, because too many false alerts create unnecessary review burden.
- Threshold tuning will be done later in `15_threshold_tuning.ipynb`.
- Final decision logic for `BLOCK`, `REVIEW`, and `APPROVE` will be defined later.

The notebook uses a holdout test set and `StratifiedKFold` validation on the training split, but it does not perform threshold optimization or final decision design.


In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sys.path.append(str(Path("..").resolve()))

from src.config import PROJECT_ROOT, RANDOM_STATE, TARGET_COLUMN, TEST_SIZE
from src.features.feature_selection import get_selected_feature_names, load_feature_selection_decisions

NOTEBOOK_NAME = "13_model_training"
SELECTED_DATA_FILE = PROJECT_ROOT / "data" / "processed" / "creditcard_selected_features.csv"
NOTEBOOK_TABLES_DIR = PROJECT_ROOT / "reports" / "tables" / NOTEBOOK_NAME
NOTEBOOK_FIGURES_DIR = PROJECT_ROOT / "reports" / "figures" / NOTEBOOK_NAME
NOTEBOOK_ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / NOTEBOOK_NAME
N_SPLITS = 5

NOTEBOOK_TABLES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("Modeling data  :", SELECTED_DATA_FILE)
print("Tables dir     :", NOTEBOOK_TABLES_DIR)
print("Artifacts dir  :", NOTEBOOK_ARTIFACTS_DIR)
print("K-fold splits  :", N_SPLITS)


Modeling data  : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/data/processed/creditcard_selected_features.csv
Tables dir     : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/13_model_training
Artifacts dir  : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/13_model_training
K-fold splits  : 5


## 1. Objectives

- Load the finalized selected-feature dataset from notebook `10_feature_selection`.
- Create one untouched holdout test split for final model evaluation.
- Apply `StratifiedKFold` only on the training split to compare baseline models fairly.
- Retrain each baseline model on the full training split after cross-validation.
- Save fitted models, cross-validation summaries, and holdout probabilities for notebook `14_model_evaluation`.

## Output Guide

This notebook writes its main outputs to:

- `reports/tables/13_model_training/`
- `artifacts/13_model_training/`


## 2. Load Final Modeling Data

The baseline models should use only the finalized selected features plus the target column.


In [2]:
df = pd.read_csv(SELECTED_DATA_FILE)
selected_features = get_selected_feature_names(load_feature_selection_decisions())

expected_columns = selected_features + [TARGET_COLUMN]
missing_columns = sorted(set(expected_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Selected dataset is missing expected columns: {missing_columns}")

model_df = df.loc[:, expected_columns].copy()
model_df["transaction_id"] = np.arange(len(model_df))

print(f"Selected dataset shape: {model_df.shape}")
print(f"Selected feature count: {len(selected_features)}")
print(f"Fraud rate: {model_df[TARGET_COLUMN].mean():.6f}")
model_df.head()


Selected dataset shape: (283726, 15)
Selected feature count: 13
Fraud rate: 0.001667


,V14_V12_interaction,V14,V17_V16_interaction,V12,V17,V10,V4,V16,V3,V11,V7,V18,log_amount,Class,transaction_id
0,0.192241,-0.311169,-0.097830,-0.617801,0.207971,0.090794,1.378155,-0.470401,2.536347,-0.551600,0.239599,0.025791,5.014760,0,0
1,-0.153151,-0.143772,-0.053260,1.065235,-0.114805,-0.166974,0.448154,0.463917,0.166480,1.612727,-0.078803,-0.183361,1.305626,0,1
2,-0.010966,-0.165946,-3.207904,0.066084,1.109969,0.207643,0.379780,-2.890083,1.773209,0.624501,0.791461,-0.121359,5.939276,0,2
3,-0.051316,-0.287924,0.724897,0.178228,-0.684093,-0.054952,-0.863291,-1.059647,1.792993,-0.226487,0.237609,1.965775,4.824306,0,3
4,-0.602601,-1.119670,0.107008,0.538196,-0.237033,0.753074,0.403034,-0.451449,1.548718,-0.822843,0.592941,-0.038195,4.262539,0,4


## 3. Define Inputs and Target

The notebook keeps the modeling inputs explicit so downstream evaluation can trace exactly which features were used.


In [3]:
X = model_df[selected_features].copy()
y = model_df[TARGET_COLUMN].copy()
row_ids = model_df["transaction_id"].copy()

input_summary = pd.DataFrame(
    [
        {"metric": "row_count", "value": int(len(model_df))},
        {"metric": "feature_count", "value": int(len(selected_features))},
        {"metric": "target_column", "value": TARGET_COLUMN},
        {"metric": "fraud_rate", "value": float(y.mean())},
    ]
)
input_summary.to_csv(NOTEBOOK_TABLES_DIR / "input_data_summary.csv", index=False)
input_summary


,metric,value
0,row_count,283726
1,feature_count,13
2,target_column,Class
3,fraud_rate,0.001667


## 4. Holdout Train/Test Split

The test set is created once and kept untouched during baseline model comparison. All K-fold validation happens only inside the training split.


In [4]:
X_train, X_test, y_train, y_test, row_id_train, row_id_test = train_test_split(
    X,
    y,
    row_ids,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "row_count": int(len(X_train)),
            "fraud_count": int(y_train.sum()),
            "fraud_rate": float(y_train.mean()),
        },
        {
            "split": "test",
            "row_count": int(len(X_test)),
            "fraud_count": int(y_test.sum()),
            "fraud_rate": float(y_test.mean()),
        },
    ]
)
split_summary.to_csv(NOTEBOOK_TABLES_DIR / "train_test_split_summary.csv", index=False)
split_summary


,split,row_count,fraud_count,fraud_rate
0,train,226980,378,0.001665
1,test,56746,95,0.001674


## 5. Baseline Preprocessing and Validation Setup

- Logistic Regression uses imputation plus scaling because coefficient-based models are sensitive to feature scale.
- Random Forest keeps a lighter preprocessing path because tree-based models do not require scaling.
- `StratifiedKFold` is applied only to `X_train` and `y_train`.

### Why Train-Only Preprocessing Prevents Data Leakage

Preprocessing steps such as imputation and scaling must be learned from the training data only. If the scaler is fit on the full dataset before the train/test split, information from the holdout test set leaks into model training through the feature means and standard deviations. That makes downstream metrics look better than they really are. In this notebook, the split happens first, the Logistic Regression scaler is fit only on `X_train`, and `X_test` is transformed using the already-fitted training scaler.


In [5]:
numeric_features = selected_features.copy()

# Final holdout workflow: fit preprocessing on the training split only.
logreg_imputer = SimpleImputer(strategy="median")
X_train_logreg_imputed = pd.DataFrame(
    logreg_imputer.fit_transform(X_train),
    columns=numeric_features,
    index=X_train.index,
)
X_test_logreg_imputed = pd.DataFrame(
    logreg_imputer.transform(X_test),
    columns=numeric_features,
    index=X_test.index,
)

logreg_scaler = StandardScaler()
X_train_logreg_scaled = pd.DataFrame(
    logreg_scaler.fit_transform(X_train_logreg_imputed),
    columns=numeric_features,
    index=X_train.index,
)
X_test_logreg_scaled = pd.DataFrame(
    logreg_scaler.transform(X_test_logreg_imputed),
    columns=numeric_features,
    index=X_test.index,
)

# Cross-validation workflow: each fold fits its own preprocessing on the fold training partition.
logreg_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        )
    ],
    remainder="drop",
)

rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            numeric_features,
        )
    ],
    remainder="drop",
)

cv_strategy = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

preprocessing_summary = pd.DataFrame(
    [
        {
            "model_name": "logistic_regression",
            "preprocessing": "median_imputation + standard_scaling fitted on X_train only",
            "validation": f"StratifiedKFold(n_splits={N_SPLITS}) on training split with fold-specific preprocessing",
        },
        {
            "model_name": "random_forest",
            "preprocessing": "median_imputation fitted on training data only",
            "validation": f"StratifiedKFold(n_splits={N_SPLITS}) on training split with fold-specific preprocessing",
        },
    ]
)
preprocessing_summary.to_csv(NOTEBOOK_TABLES_DIR / "preprocessing_summary.csv", index=False)
preprocessing_summary


,model_name,preprocessing,validation
0,logistic_regression,median_imputation + standard_scaling fitted on...,StratifiedKFold(n_splits=5) on training split ...
1,random_forest,median_imputation fitted on training data only,StratifiedKFold(n_splits=5) on training split ...


## 6. Cross-Validation Helpers

The helper below trains each baseline model across the training folds, records fold-level metrics, and returns out-of-fold probabilities for later inspection. Because the helper clones the preprocessing pipeline inside each fold, the imputer and scaler are fit only on that fold's training partition before being applied to the fold validation partition.


In [6]:
def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
    }


def run_training_cv(model_name, pipeline, X_train, y_train, row_id_train, cv_strategy):
    fold_rows = []
    oof_rows = []

    for fold_number, (fit_idx, valid_idx) in enumerate(cv_strategy.split(X_train, y_train), start=1):
        X_fit = X_train.iloc[fit_idx]
        X_valid = X_train.iloc[valid_idx]
        y_fit = y_train.iloc[fit_idx]
        y_valid = y_train.iloc[valid_idx]
        row_id_valid = row_id_train.iloc[valid_idx]

        fold_model = clone(pipeline)
        fold_model.fit(X_fit, y_fit)
        valid_prob = fold_model.predict_proba(X_valid)[:, 1]

        fold_metrics = compute_binary_metrics(y_valid, valid_prob)
        fold_metrics.update(
            {
                "model_name": model_name,
                "fold": fold_number,
                "validation_rows": int(len(X_valid)),
                "validation_fraud_count": int(y_valid.sum()),
            }
        )
        fold_rows.append(fold_metrics)

        oof_rows.append(
            pd.DataFrame(
                {
                    "transaction_id": row_id_valid.to_numpy(),
                    "y_true": y_valid.to_numpy(),
                    "model_name": model_name,
                    "fold": fold_number,
                    "predicted_probability": valid_prob,
                    "predicted_label_0_5": (valid_prob >= 0.5).astype(int),
                }
            )
        )

    fold_metrics_df = pd.DataFrame(fold_rows)
    oof_predictions_df = pd.concat(oof_rows, ignore_index=True).sort_values(["fold", "transaction_id"]).reset_index(drop=True)

    summary_df = pd.DataFrame(
        [
            {
                "model_name": model_name,
                "metric_scope": "cv_mean",
                "precision": fold_metrics_df["precision"].mean(),
                "recall": fold_metrics_df["recall"].mean(),
                "f1": fold_metrics_df["f1"].mean(),
                "roc_auc": fold_metrics_df["roc_auc"].mean(),
                "pr_auc": fold_metrics_df["pr_auc"].mean(),
            },
            {
                "model_name": model_name,
                "metric_scope": "cv_std",
                "precision": fold_metrics_df["precision"].std(ddof=0),
                "recall": fold_metrics_df["recall"].std(ddof=0),
                "f1": fold_metrics_df["f1"].std(ddof=0),
                "roc_auc": fold_metrics_df["roc_auc"].std(ddof=0),
                "pr_auc": fold_metrics_df["pr_auc"].std(ddof=0),
            },
        ]
    )

    return fold_metrics_df, oof_predictions_df, summary_df


## 7. Baseline Model 1: Logistic Regression

This is the linear baseline. `class_weight='balanced'` helps the model pay more attention to the rare fraud class during fitting. The model is trained only on the training split, then used to generate both class predictions and fraud probabilities on the untouched test set.


In [7]:
logreg_pipeline = Pipeline(
    steps=[
        ("preprocessor", logreg_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                solver="lbfgs",
            ),
        ),
    ]
)

logreg_fold_metrics, logreg_oof_predictions, logreg_cv_summary = run_training_cv(
    model_name="logistic_regression",
    pipeline=logreg_pipeline,
    X_train=X_train,
    y_train=y_train,
    row_id_train=row_id_train,
    cv_strategy=cv_strategy,
)

logreg_fold_metrics


,precision,recall,f1,roc_auc,pr_auc,model_name,fold,validation_rows,validation_fraud_count
0,0.056208,0.893333,0.105762,0.983064,0.750033,logistic_regression,1,45396,75
1,0.051166,0.906667,0.096866,0.973249,0.724343,logistic_regression,2,45396,75
2,0.062880,0.815789,0.116761,0.973395,0.652893,logistic_regression,3,45396,76
3,0.059272,0.921053,0.111376,0.987944,0.725148,logistic_regression,4,45396,76
4,0.060481,0.960526,0.113796,0.987174,0.745364,logistic_regression,5,45396,76


## 8. Baseline Model 2: Random Forest

This is the nonlinear tree baseline.

### Why Random Forest Is Useful Here

- It can capture nonlinear fraud patterns that a linear model may miss.
- It gives a stronger comparison against Logistic Regression.
- It is still acceptable as a baseline tree-based model.


In [8]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

rf_fold_metrics, rf_oof_predictions, rf_cv_summary = run_training_cv(
    model_name="random_forest",
    pipeline=random_forest_pipeline,
    X_train=X_train,
    y_train=y_train,
    row_id_train=row_id_train,
    cv_strategy=cv_strategy,
)

rf_fold_metrics


,precision,recall,f1,roc_auc,pr_auc,model_name,fold,validation_rows,validation_fraud_count
0,0.950000,0.760000,0.844444,0.938534,0.819631,random_forest,1,45396,75
1,0.968254,0.813333,0.884058,0.945653,0.868549,random_forest,2,45396,75
2,0.927273,0.671053,0.778626,0.906206,0.754614,random_forest,3,45396,76
3,0.897059,0.802632,0.847222,0.953025,0.859088,random_forest,4,45396,76
4,0.941176,0.842105,0.888889,0.952996,0.847412,random_forest,5,45396,76


## 9. Cross-Validation Comparison Summary

This summary compares fold-level mean and variability across the two baseline models. The test set is still untouched at this stage.


In [9]:
cv_summary = pd.concat([logreg_cv_summary, rf_cv_summary], ignore_index=True)
cv_fold_metrics = pd.concat([logreg_fold_metrics, rf_fold_metrics], ignore_index=True)
cv_oof_predictions = pd.concat([logreg_oof_predictions, rf_oof_predictions], ignore_index=True)

cv_summary.to_csv(NOTEBOOK_TABLES_DIR / "cv_summary.csv", index=False)
cv_fold_metrics.to_csv(NOTEBOOK_TABLES_DIR / "cv_fold_metrics.csv", index=False)
cv_oof_predictions.to_csv(NOTEBOOK_TABLES_DIR / "cv_oof_predictions.csv", index=False)

cv_summary


,model_name,metric_scope,precision,recall,f1,roc_auc,pr_auc
0,logistic_regression,cv_mean,0.058001,0.899474,0.108912,0.980965,0.719556
1,logistic_regression,cv_std,0.004037,0.047509,0.007022,0.006458,0.034909
2,random_forest,cv_mean,0.936752,0.777825,0.848648,0.939283,0.829859
3,random_forest,cv_std,0.023887,0.059542,0.039487,0.017390,0.041051


## 10. Fit Final Baseline Models on the Full Training Split

After comparing the models across training folds, each baseline model is retrained on the full training split so notebook `14_model_evaluation` can score them once on the untouched holdout test set. For Logistic Regression, the imputer and scaler used here were fit only on `X_train` in the earlier preprocessing section and are now reused to transform `X_test` without refitting.


In [10]:
final_logreg_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    solver="lbfgs",
)
final_random_forest_pipeline = clone(random_forest_pipeline)

final_logreg_model.fit(X_train_logreg_scaled, y_train)
final_random_forest_pipeline.fit(X_train, y_train)

logreg_test_predictions = final_logreg_model.predict(X_test_logreg_scaled)
logreg_test_probabilities = final_logreg_model.predict_proba(X_test_logreg_scaled)[:, 1]
rf_test_predictions = final_random_forest_pipeline.predict(X_test)
rf_test_probabilities = final_random_forest_pipeline.predict_proba(X_test)[:, 1]

holdout_predictions = pd.DataFrame(
    {
        "transaction_id": row_id_test.to_numpy(),
        "y_true": y_test.to_numpy(),
        "logistic_regression_probability": logreg_test_probabilities,
        "random_forest_probability": rf_test_probabilities,
    }
).sort_values("transaction_id").reset_index(drop=True)

holdout_predictions["logistic_regression_prediction_0_5"] = logreg_test_predictions
holdout_predictions["random_forest_prediction_0_5"] = rf_test_predictions

holdout_predictions.to_csv(NOTEBOOK_TABLES_DIR / "baseline_test_predictions.csv", index=False)
holdout_predictions.head()


,transaction_id,y_true,logistic_regression_probability,random_forest_probability,logistic_regression_prediction_0_5,random_forest_prediction_0_5
0,5,0,0.042209,0.0,0,0
1,9,0,0.039224,0.0,0,0
2,10,0,0.015697,0.0,0,0
3,13,0,0.189465,0.0,0,0
4,15,0,0.019063,0.0,0,0


## 11. Logistic Regression Holdout Evaluation

This section evaluates the Logistic Regression baseline on the untouched test set using both hard predictions and prediction probabilities.

### Metric Interpretation

- Recall shows how many fraud cases were caught.
- Precision shows how many predicted fraud cases were actually fraud.
- In fraud detection, recall is usually more important than accuracy because missing fraud is often more costly than reviewing extra alerts.


In [11]:
logreg_confusion_matrix = confusion_matrix(
    y_test,
    holdout_predictions["logistic_regression_prediction_0_5"],
)

logreg_holdout_metrics = pd.DataFrame(
    [
        {
            "model_name": "logistic_regression",
            "evaluation_split": "holdout_test",
            "precision": precision_score(
                y_test,
                holdout_predictions["logistic_regression_prediction_0_5"],
                zero_division=0,
            ),
            "recall": recall_score(
                y_test,
                holdout_predictions["logistic_regression_prediction_0_5"],
                zero_division=0,
            ),
            "f1_score": f1_score(
                y_test,
                holdout_predictions["logistic_regression_prediction_0_5"],
                zero_division=0,
            ),
            "roc_auc": roc_auc_score(
                y_test,
                holdout_predictions["logistic_regression_probability"],
            ),
            "pr_auc": average_precision_score(
                y_test,
                holdout_predictions["logistic_regression_probability"],
            ),
            "tn": int(logreg_confusion_matrix[0, 0]),
            "fp": int(logreg_confusion_matrix[0, 1]),
            "fn": int(logreg_confusion_matrix[1, 0]),
            "tp": int(logreg_confusion_matrix[1, 1]),
        }
    ]
)

logreg_holdout_metrics.to_csv(
    NOTEBOOK_TABLES_DIR / "logistic_regression_holdout_metrics.csv",
    index=False,
)
pd.DataFrame(
    logreg_confusion_matrix,
    index=["actual_0", "actual_1"],
    columns=["predicted_0", "predicted_1"],
)


,predicted_0,predicted_1
actual_0,55294,1357
actual_1,13,82


## 12. Random Forest Holdout Evaluation

This section evaluates the Random Forest baseline on the untouched test set using both hard predictions and prediction probabilities.


In [12]:
rf_confusion_matrix = confusion_matrix(
    y_test,
    holdout_predictions["random_forest_prediction_0_5"],
)

rf_holdout_metrics = pd.DataFrame(
    [
        {
            "model_name": "random_forest",
            "evaluation_split": "holdout_test",
            "precision": precision_score(
                y_test,
                holdout_predictions["random_forest_prediction_0_5"],
                zero_division=0,
            ),
            "recall": recall_score(
                y_test,
                holdout_predictions["random_forest_prediction_0_5"],
                zero_division=0,
            ),
            "f1_score": f1_score(
                y_test,
                holdout_predictions["random_forest_prediction_0_5"],
                zero_division=0,
            ),
            "roc_auc": roc_auc_score(
                y_test,
                holdout_predictions["random_forest_probability"],
            ),
            "pr_auc": average_precision_score(
                y_test,
                holdout_predictions["random_forest_probability"],
            ),
            "tn": int(rf_confusion_matrix[0, 0]),
            "fp": int(rf_confusion_matrix[0, 1]),
            "fn": int(rf_confusion_matrix[1, 0]),
            "tp": int(rf_confusion_matrix[1, 1]),
        }
    ]
)

rf_holdout_metrics.to_csv(
    NOTEBOOK_TABLES_DIR / "random_forest_holdout_metrics.csv",
    index=False,
)
pd.DataFrame(
    rf_confusion_matrix,
    index=["actual_0", "actual_1"],
    columns=["predicted_0", "predicted_1"],
)


,predicted_0,predicted_1
actual_0,56648,3
actual_1,25,70


## 13. Save Training Outputs

Notebook `14_model_evaluation` should not need to retrain these models. This section saves the fitted pipelines, split metadata, and holdout probabilities needed for comparison.


In [13]:
split_indices = {
    "train_transaction_ids": row_id_train.tolist(),
    "test_transaction_ids": row_id_test.tolist(),
}
(NOTEBOOK_ARTIFACTS_DIR / "train_test_split_ids.json").write_text(
    json.dumps(split_indices, indent=2),
    encoding="utf-8",
)

joblib.dump(
    {
        "model": final_logreg_model,
        "imputer": logreg_imputer,
        "scaler": logreg_scaler,
        "model_name": "logistic_regression",
        "selected_features": selected_features,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "cv_splits": N_SPLITS,
    },
    NOTEBOOK_ARTIFACTS_DIR / "baseline_logistic_regression.joblib",
)

joblib.dump(
    {
        "model": final_random_forest_pipeline,
        "model_name": "random_forest",
        "selected_features": selected_features,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "cv_splits": N_SPLITS,
    },
    NOTEBOOK_ARTIFACTS_DIR / "baseline_random_forest.joblib",
)

artifact_manifest = pd.DataFrame(
    [
        {"artifact_type": "model", "path": str(NOTEBOOK_ARTIFACTS_DIR / "baseline_logistic_regression.joblib")},
        {"artifact_type": "model", "path": str(NOTEBOOK_ARTIFACTS_DIR / "baseline_random_forest.joblib")},
        {"artifact_type": "cv_summary", "path": str(NOTEBOOK_TABLES_DIR / "cv_summary.csv")},
        {"artifact_type": "cv_fold_metrics", "path": str(NOTEBOOK_TABLES_DIR / "cv_fold_metrics.csv")},
        {"artifact_type": "cv_oof_predictions", "path": str(NOTEBOOK_TABLES_DIR / "cv_oof_predictions.csv")},
        {"artifact_type": "holdout_test_predictions", "path": str(NOTEBOOK_TABLES_DIR / "baseline_test_predictions.csv")},
        {"artifact_type": "logreg_holdout_metrics", "path": str(NOTEBOOK_TABLES_DIR / "logistic_regression_holdout_metrics.csv")},
        {"artifact_type": "rf_holdout_metrics", "path": str(NOTEBOOK_TABLES_DIR / "random_forest_holdout_metrics.csv")},
        {"artifact_type": "split_ids", "path": str(NOTEBOOK_ARTIFACTS_DIR / "train_test_split_ids.json")},
    ]
)
artifact_manifest.to_csv(NOTEBOOK_TABLES_DIR / "training_artifact_manifest.csv", index=False)
artifact_manifest


,artifact_type,path
0,model,/Users/mohammadmubashir/VCode/Credit-Card-Frau...
1,model,/Users/mohammadmubashir/VCode/Credit-Card-Frau...
2,cv_summary,/Users/mohammadmubashir/VCode/Credit-Card-Frau...
3,cv_fold_metrics,/Users/mohammadmubashir/VCode/Credit-Card-Frau...
4,cv_oof_predictions,/Users/mohammadmubashir/VCode/Credit-Card-Frau...
5,holdout_test_predictions,/Users/mohammadmubashir/VCode/Credit-Card-Frau...
6,logreg_holdout_metrics,/Users/mohammadmubashir/VCode/Credit-Card-Frau...
7,rf_holdout_metrics,/Users/mohammadmubashir/VCode/Credit-Card-Frau...
8,split_ids,/Users/mohammadmubashir/VCode/Credit-Card-Frau...


## 14. Training Summary

This notebook ends with a compact handoff summary. Detailed comparison belongs in notebook `14_model_evaluation`.


In [14]:
training_overview = pd.DataFrame(
    [
        {
            "model_name": "logistic_regression",
            "validation_strategy": f"{N_SPLITS}-fold StratifiedKFold on training split",
            "final_fit_status": "trained_on_full_training_split",
            "holdout_test_rows": int(len(holdout_predictions)),
        },
        {
            "model_name": "random_forest",
            "validation_strategy": f"{N_SPLITS}-fold StratifiedKFold on training split",
            "final_fit_status": "trained_on_full_training_split",
            "holdout_test_rows": int(len(holdout_predictions)),
        },
    ]
)
training_overview.to_csv(NOTEBOOK_TABLES_DIR / "training_overview.csv", index=False)

training_report = f"""# Model Training Report

## Purpose
- Train the baseline fraud-detection models on the finalized selected-feature dataset.
- Keep the holdout test set untouched during cross-validation.
- Save the trained artifacts and holdout probabilities for notebook `14_model_evaluation`.

## Training Inputs
- Modeling dataset: `{SELECTED_DATA_FILE}`
- Selected feature count: `{len(selected_features)}`
- Total rows: `{len(model_df)}`
- Fraud rate: `{y.mean():.6f}`
- Test size: `{TEST_SIZE}`
- Random state: `{RANDOM_STATE}`
- Cross-validation: `StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={RANDOM_STATE})` on the training split only

## Models Trained
- Logistic Regression with median imputation, standard scaling, `max_iter=1000`, and `class_weight='balanced'`
- Random Forest with median imputation, `n_estimators=100`, and `class_weight='balanced'`

## Handoff to Next Notebook
- Read `cv_summary.csv` and `cv_fold_metrics.csv` to compare fold performance.
- Read `baseline_test_predictions.csv` to evaluate both final baseline models once on the untouched holdout test set.
- Read `logistic_regression_holdout_metrics.csv` for the baseline Logistic Regression confusion matrix and core holdout metrics.
- Read `random_forest_holdout_metrics.csv` for the baseline Random Forest confusion matrix and core holdout metrics.
"""

report_path = NOTEBOOK_TABLES_DIR / "model_training_report.md"
report_path.write_text(training_report, encoding="utf-8")

print(f"Training report saved to: {report_path}")
training_overview


Training report saved to: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/13_model_training/model_training_report.md


,model_name,validation_strategy,final_fit_status,holdout_test_rows
0,logistic_regression,5-fold StratifiedKFold on training split,trained_on_full_training_split,56746
1,random_forest,5-fold StratifiedKFold on training split,trained_on_full_training_split,56746
